# 극한 — '도착'이 아니라 '접근'

> 미적분 2강 · 극한

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [극한 — '도착'이 아니라 '접근'](https://mioon1402.github.io/timeseriesdata/calc/C02-limit.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 극한이 필요한가

## 1. 다가가는 수들

## 2. lim 기호 읽는 법

## 3. 확대의 비유 — 곡선이 직선이 된다

## 4. 극한값은 함숫값이 아니다

## 5. 극한이 없을 때 — 점프와 진동

## 6. 0.999… 는 1인가

## 7. 파이썬으로 확인하기

**2-1. 다가가는 수들**

In [ ]:
import numpy as np

print(f"{'n':>3} {'항':>22} {'1과의 차':>12}")
for n in range(1, 17):
    항 = 1 - 10.0**(-n)
    print(f"{n:>3} {항:>22.16f} {10.0**(-n):>12.0e}")

print("\n→ 어떤 항도 1이 아니다. 그런데 '1과의 차'를 원하는 만큼 작게 만들 수 있다.")
print("  (16번째부터는 컴퓨터의 소수 자릿수 한계로 1로 보인다 — 수학이 아니라 기계의 한계다)")

**2-2. 확대하면 직선이 되는가**

In [ ]:
def 어긋남(f, x0, 반폭, 기울기):
    """창 안에서 곡선이 접선에서 벗어난 최대 거리를, 창 높이 대비 %로"""
    d = np.linspace(-반폭, 반폭, 2001)
    곡선 = f(x0 + d)
    접선 = f(x0) + 기울기 * d
    return 100 * np.abs(곡선 - 접선).max() / (2 * 반폭)

함수들 = {
    "x²":  (lambda x: x**2,      lambda x: 2*x),
    "sin": (np.sin,              np.cos),
    "√x":  (np.sqrt,             lambda x: 0.5/np.sqrt(x)),
    "|x|": (np.abs,              lambda x: 0.0),      # x=0 에서는 접선이 없다
}

print(f"{'창 반폭':>8}", "".join(f"{이름:>10}" for 이름 in 함수들))
for w in [1.0, 0.1, 0.01, 0.001]:
    행 = f"{w:>8}"
    for 이름, (f, df) in 함수들.items():
        x0 = 0.0 if 이름 == "|x|" else 1.0
        행 += f"{어긋남(f, x0, w, df(x0)):>9.4f}%"
    print(행)

print("\n→ 매끈한 함수는 확대할수록 어긋남이 10배씩 줄어든다.")
print("  |x| 는 x=0 에서 50% 로 고정 — 아무리 확대해도 꺾인 채다.")

**2-3. 구멍이 뚫린 함수**

In [ ]:
def f(x):
    return (x**2 - 1) / (x - 1)        # x=1 에서 0/0

print("x = 1 에 왼쪽에서 다가가기")
for d in [0.1, 0.01, 0.001, 0.0001]:
    print(f"  x = {1-d:<10.4f} f(x) = {f(1-d):.6f}")

print("\nx = 1 에 오른쪽에서 다가가기")
for d in [0.1, 0.01, 0.001, 0.0001]:
    print(f"  x = {1+d:<10.4f} f(x) = {f(1+d):.6f}")

print("\n→ 양쪽 모두 2를 가리킨다.  lim(x→1) f(x) = 2")

try:
    print("f(1) =", f(1))
except ZeroDivisionError:
    print("f(1) = 계산 불가 (0/0)  ← 함숫값은 없는데 극한은 있다")

**2-4. 왜 2인가 — 약분해보면**

In [ ]:
import sympy as sp

x = sp.Symbol('x')
식 = (x**2 - 1) / (x - 1)
print("원래 식 :", 식)
print("약분하면:", sp.simplify(식), "   ← x ≠ 1 인 곳에서는 이것과 완전히 같다")
print("극한    :", sp.limit(식, x, 1))
print()
print("(x²-1) = (x-1)(x+1) 이므로 x≠1 이면 (x-1) 이 약분되어 x+1 이 남는다.")
print("x=1 을 '대입'한 게 아니라, x≠1 인 점들만 보고 목적지를 읽은 것이다.")

**2-5. 극한이 없는 두 가지**

In [ ]:
# ① 점프 — 좌우 극한이 다르다
def 점프(x): return np.where(x < 1, x, x + 1.2)
print("점프 함수")
print("  왼쪽에서 :", [round(float(점프(np.array(1-d))), 4) for d in [0.1, 0.01, 0.001]])
print("  오른쪽에서:", [round(float(점프(np.array(1+d))), 4) for d in [0.1, 0.01, 0.001]])
print("  → 1 과 2.2. 서로 다르므로 극한이 없다\n")

# ② 진동 — 어디에도 정착하지 않는다
def 진동(x): return np.sin(1/(x-1))
print("진동 함수 sin(1/(x-1))")
for d in [0.1, 0.01, 0.001, 0.0001, 0.00001]:
    print(f"  x = 1+{d:<9} f = {진동(1+d):>9.5f}")
print("  → 값이 -1 과 1 사이를 계속 오간다. 목적지가 없다")

**2-6. 연습문제**

In [ ]:
# 문제 1. lim(x→0) (1 - cos x) / x²  을 수치로 추정해보세요. (x = 0.1, 0.01, 0.001)

# 문제 2. |x| 대신 x·|x| 를 x=0 에서 확대하면 직선이 될까요?
#         2-2 셀의 '어긋남' 함수로 확인해보세요. (기울기 0 으로)

# 문제 3. 1강의 고리 합을 n = 10, 100, 1000, 10000 으로 계산해
#         "극한이 π 다" 를 수치로 확인해보세요.

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
# 문제 1 — 1/2 로 간다
print("문제 1")
for x in [0.1, 0.01, 0.001, 0.0001]:
    print(f"  x={x:<8} (1-cos x)/x² = {(1-np.cos(x))/x**2:.8f}")
print("  → 0.5 (11강의 테일러 급수로 보면 cos x ≈ 1 - x²/2 이므로 당연하다)\n")

# 문제 2 — x|x| 는 x=0 에서 매끈하다 (미분 가능, 기울기 0)
print("문제 2")
for w in [1.0, 0.1, 0.01]:
    print(f"  창 반폭 {w:<6} 어긋남 = {어긋남(lambda x: x*np.abs(x), 0.0, w, 0.0):.4f}%")
print("  → 줄어든다. x|x| 는 꺾이지 않고 매끈하게 휘어서 지나간다\n")

# 문제 3 — 고리 합의 극한
print("문제 3")
for n in [10, 100, 1000, 10000]:
    dr = 1.0/n
    r = np.arange(n) * dr
    print(f"  n={n:>6}  합 = {(2*np.pi*r*dr).sum():.8f}")
print(f"  참값 π = {np.pi:.8f}   ← 극한이 π 다")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)